[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_1_Elimination_of_Constants.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 📡 Lab 1: Derivation of PDEs by Elimination of Constants
### *Mission Control: The Terrain Scanner*
**Course:** Applied Partial Differential Equations & Planetary Robotics  
**Syllabus Topic:** Chapter 1, Section 1.2 (Pages 6–11)

---
### 🧭 The Rosetta Stone: Translating Math into English and Code
| 1. Professor's Blackboard Math | 2. Plain Human English | 3. Python Code |
| :--- | :--- | :--- |
| $p = \frac{\partial z}{\partial x}, \quad q = \frac{\partial z}{\partial y}$ | "The tilt of the terrain Eastward, and Northward." | `P, Q = np.gradient(Z, dx, dy)` |
| $F(x, y, z, a, b) = 0$ | "A family of surfaces with parameters $a$ and $b$." | `Z = (X - a)**2 + (Y - b)**2` |
| $f(x, y, z, p, q) = 0$ | "The universal slope law connecting height and tilt." | `residual = (P**2 + Q**2) - 4*Z` |

---
## 1. In Plain English: "The Sculptor's Mold"
Imagine you are 3D-printing a landscape of sand dunes on Mars. Each dune might have a different height $a$ and a different location $b$.
If you drop an autonomous rover on that terrain, the rover does not care about the numbers $a$ and $b$; it only cares:  
> *"What is the physical slope under my wheels right now? Is it too steep to climb?"*

**Eliminating constants** is the mathematical process of removing the parameters $a$ and $b$ to find the **universal physical law** that governs the slope everywhere, regardless of the crater's size or position.

---
## 2. The "WHY": Why do we do this method? What does it help with?
* **Why do we do it?** Physical laws (like gravity or traction) do not change because someone labeled a crater with parameters $a$ and $b$.
* **What it helps with:** It allows the rover's flight software to verify terrain safety from local slope sensors alone, without needing prior map parameters.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab1

# Ensure inline plotting
%matplotlib inline
print("Lab 1 Environment Initialized! Ready for Mission Control.")


---
## 3. The Mathematical Recipe & Worked Textbook Example

### The 3 Situations:
1. **Situation (1):** Number of arbitrary constants $<$ independent variables $\implies$ More than one 1st-order PDE.
2. **Situation (2):** Number of arbitrary constants $=$ independent variables $\implies$ Unique 1st-order PDE.
3. **Situation (3):** Number of arbitrary constants $>$ independent variables $\implies$ Higher-order PDE.

### Worked Example (Page 8 of your lecture notes):
Given the parabolic surface:
$$z = a x + b y + a^2 + b^2$$
* **Step 1:** Differentiate w.r.t $x$:  
  $$\frac{\partial z}{\partial x} = p = a$$
* **Step 2:** Differentiate w.r.t $y$:  
  $$\frac{\partial z}{\partial y} = q = b$$
* **Step 3:** Substitute $a = p$ and $b = q$ back into the original relation:
  $$z = x p + y q + p^2 + q^2$$
* **Result:** A unique non-linear first-order PDE free from constants!


---
## 4. 📝 Paper & Pen Challenge 1 (Textbook Problem)

> 💡 **Instructions:** Grab a piece of paper and a pen. Work through the steps below.  
> *(If you get stuck at any point, open `INSTRUCTOR_ANSWER_KEY.md` Section 1.1 for full worked steps!)*

**Problem:** Given the family of circular craters:
$$z = (x - a)^2 + (y - b)^2$$
1. Differentiate partially with respect to $x$: $p = \frac{\partial z}{\partial x} = 2(x - a) \implies (x - a) = \frac{p}{2}$.
2. Differentiate partially with respect to $y$: $q = \frac{\partial z}{\partial y} = 2(y - b) \implies (y - b) = \frac{q}{2}$.
3. Square both derived equations and add them together.
4. Substitute $z$ into the result.

Enter your resulting PDE string (e.g. `"p^2 + q^2 = 4z"`) in the cell below:


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ### (≈ 1 line of code)
# Enter your resulting PDE string from your paper calculations (e.g. "p^2 + q^2 = 4z")
student_pde_1 = None
### END CODE HERE ###

# Run verification test:
test_lab1.check_paper_challenge_1(student_pde_1)


---
## 5. 🚀 The Rover Bridge Challenge (Applying Paper Math to Mars)

Now let's apply this directly to the **Martian Rover Engine**!

A crater basin on Mars is modeled by the quadric surface:
$$z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2$$

**On your paper:**
1. Compute $p = \frac{\partial z}{\partial x} = a x \implies a = \frac{p}{x}$.
2. Compute $q = \frac{\partial z}{\partial y} = b y \implies b = \frac{q}{y}$.
3. Substitute $a$ and $b$ back into the equation.

What is the resulting PDE? Enter it below:


In [ ]:
# GRADED CELL: Rover Bridge Challenge
### START CODE HERE ### (≈ 1 line of code)
# Enter your resulting PDE string from your paper calculations (e.g. "x*p + y*q = 2z")
rover_basin_pde = None
### END CODE HERE ###

# Run verification test:
test_lab1.check_paper_challenge_2(rover_basin_pde)


---
## 6. 💻 Python Implementation: Terrain Slope Verification

Now let's write the flight software routine that checks whether a numerical heightmap satisfies our derived PDE:
$$\text{residual} = (p^2 + q^2) - 4z \approx 0$$


In [ ]:
# GRADED FUNCTION: verify_crater_pde
def verify_crater_pde(Z: np.ndarray, P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    # Computes the PDE residual: (p^2 + q^2) - 4*z
    ### START CODE HERE ### (≈ 1 line of code)
    # Compute residual: (P**2 + Q**2) - 4.0 * Z
    residual = None
    ### END CODE HERE ###
    return residual

# Run automated test:
test_lab1.test_eliminate_constants_fn(verify_crater_pde)


---
## 7. 🎨 Visualizing the Physics: 3D Surface & Slope Vectors

Let's plot the crater surface and see how the slope vectors $(p, q)$ point directly up the crater walls:


In [ ]:
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Z = (X - 0.5)**2 + (Y + 0.5)**2  # a=0.5, b=-0.5

dx = x[1] - x[0]
dy = y[1] - y[0]
P, Q = np.gradient(Z, dx, dy)

fig = plt.figure(figsize=(12, 5))

# 3D Surface Plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='copper', alpha=0.85)
ax1.set_title("Mars Crater Basin Surface $z(x, y)$", fontsize=12)

# 2D Slope Field (Quiver)
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contour(X, Y, Z, levels=15, cmap='copper')
ax2.quiver(X[::3, ::3], Y[::3, ::3], P[::3, ::3], Q[::3, ::3], color='red')
ax2.set_title("Terrain Slope Field $(p, q) = (\\partial z/\\partial x, \\partial z/\\partial y)$", fontsize=12)

plt.tight_layout()
plt.show()

print("Lab 1 Complete! You have mastered the derivation of PDEs by elimination of constants.")
